In [139]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, accuracy_score, classification_report ,recall_score,precision_score , f1_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
df=pd.read_csv("../data/processed-data/cleaned_data.csv")

In [140]:
X = df.drop(columns=["Churn"],
            errors="ignore")
y=df["Churn"]


In [141]:
df["Churn"].value_counts()*100/df.shape[0]

Churn
0    73.421502
1    26.578498
Name: count, dtype: float64

In [142]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=2)

In [143]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(5625, 19)
(1407, 19)
(5625,)
(1407,)


In [144]:
print(X_train[["TotalCharges", "MonthlyCharges", "tenure"]].dtypes)

TotalCharges      float64
MonthlyCharges      int64
tenure              int64
dtype: object


In [151]:
cat_cols = X_train.select_dtypes(
    include=["object", "category", "bool","str"]
).columns.tolist()
num_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()
cat_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore",drop="first"))
])
num_pipeline = Pipeline([
    ("scaler", StandardScaler())
])
preprocessor = ColumnTransformer([
    ("cat", cat_pipeline, cat_cols),
    ("num", num_pipeline, num_cols)
])

In [152]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("lr", LogisticRegression(C=1.0, max_iter=1000))
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
lr = model.named_steps["lr"]


In [153]:
#with out pca 
print("Train accuracy:", model.score(X_train, y_train))
print("Test accuracy:", model.score(X_test, y_test))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, zero_division=0))
print("Number of coefficients:", lr.coef_.size)

Train accuracy: 0.8161777777777778
Test accuracy: 0.798862828713575
Precision: 0.6444444444444445
Recall: 0.5427807486631016
F1 Score: 0.5892597968069666
Number of coefficients: 495


In [154]:
pca_model = Pipeline([
    ("preprocessor", preprocessor),
    ("poly",PolynomialFeatures(degree=2,include_bias=False)),
    ("pca", PCA(n_components=0.95)),
    ("lr1", LogisticRegression(C=4.0,max_iter=1000))
])
pca_model.fit(X_train, y_train)
y_pred_custom = pca_model.predict(X_test)
lr1 = pca_model.named_steps["lr1"]

In [155]:
# Evaluate using pca
print("Train accuracy:", pca_model.score(X_train, y_train))
print("Accuracy:", accuracy_score(y_test, y_pred_custom))
print("Precision:", precision_score(y_test, y_pred_custom))
print("Recall:", recall_score(y_test, y_pred_custom))
print("F1 score:", f1_score(y_test, y_pred_custom))
print("Number of coefficients:", lr1.coef_.size)

Train accuracy: 0.8108444444444445
Accuracy: 0.8052594171997157
Precision: 0.6655629139072847
Recall: 0.5374331550802139
F1 score: 0.5946745562130178
Number of coefficients: 90


In [150]:
print(classification_report(y_test, y_pred))
print("after pca model:")
print(classification_report(y_test, y_pred_custom))

              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1033
           1       0.63      0.54      0.58       374

    accuracy                           0.79      1407
   macro avg       0.74      0.71      0.72      1407
weighted avg       0.78      0.79      0.79      1407

after pca model:
              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1033
           1       0.66      0.55      0.60       374

    accuracy                           0.81      1407
   macro avg       0.75      0.72      0.73      1407
weighted avg       0.80      0.81      0.80      1407



## Conclusion

The Logistic Regression model without PCA achieved a test accuracy of 79.89% and an F1-score of 58.93%, using 495 coefficients.

After applying PCA, the model showed a small improvement in accuracy of 80.05% and an F1-score of 59.4%, using 90 coefficients. PCA can also reduce the number of features used by the model.

However, recall remained around 54% in both models, meaning the model still misses some customers who actually churn.

Overall, the PCA model appears slightly better based on the earlier results. Further improvements can be made by tuning the model and focusing on increasing recall to identify more customers at risk of leaving.